In [1]:
from qiskit import transpile
from qiskit.circuit.library import real_amplitudes
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer import AerSimulator

sim = AerSimulator()

### Simulating using estimator

In [2]:
from qiskit_aer.primitives import EstimatorV2

psi1 = transpile(real_amplitudes(num_qubits=2, reps=2), sim, optimization_level=0)
psi2 = transpile(real_amplitudes(num_qubits=2, reps=3), sim, optimization_level=0)

H1 = SparsePauliOp.from_list([("II", 1), ("IZ", 2), ("XI", 3)])
H2 = SparsePauliOp.from_list([("IZ", 1)])
H3 = SparsePauliOp.from_list([("ZI", 1), ("ZZ", 1)])

theta1 = [0, 1, 1, 2, 3, 5]
theta2 = [0, 1, 1, 2, 3, 5, 8, 13]
theta3 = [1, 2, 3, 4, 5, 6]

estimator = EstimatorV2()


### calculate [ [<psi1(theta1)|H1|psi1(theta1)>,
###              <psi1(theta3)|H3|psi1(theta3)>],
###             [<psi2(theta2)|H2|psi2(theta2)>] ]

In [3]:
job = estimator.run(
    [
        (psi1, [H1, H3], [theta1, theta3]),
        (psi2, H2, theta2)
    ],
    precision=0.01
)
result = job.result()
print(f"expectation values : psi1 = {result[0].data.evs}, psi2 = {result[1].data.evs}")

expectation values : psi1 = [ 1.56093961 -1.08901856], psi2 = 0.16639239364539682


### Simulating using sampler

In [4]:
from qiskit_aer.primitives import SamplerV2
from qiskit import QuantumCircuit

# create a Bell circuit
bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
bell.measure_all()

# create two parameterized circuits
pqc = real_amplitudes(num_qubits=2, reps=2)
pqc.measure_all()
pqc = transpile(pqc, sim, optimization_level=0)
pqc2 = real_amplitudes(num_qubits=2, reps=3)
pqc2.measure_all()
pqc2 = transpile(pqc2, sim, optimization_level=0)

theta1 = [0, 1, 1, 2, 3, 5]
theta2 = [0, 1, 2, 3, 4, 5, 6, 7]

# initialization of the sampler
sampler = SamplerV2()

# collect 128 shots from the Bell circuit
job = sampler.run([bell], shots=128)
job_result = job.result()
print(f"counts for Bell circuit : {job_result[0].data.meas.get_counts()}")
 
# run a sampler job on the parameterized circuits
job2 = sampler.run([(pqc, theta1), (pqc2, theta2)])
job_result = job2.result()
print(f"counts for parameterized circuit : {job_result[0].data.meas.get_counts()}")

counts for Bell circuit : {'00': 61, '11': 67}
counts for parameterized circuit : {'11': 423, '00': 136, '01': 387, '10': 78}


### Simulating with noise model from actual hardware

In [10]:
from qiskit_ibm_runtime import QiskitRuntimeService
provider = QiskitRuntimeService(name="main_school", channel="ibm_quantum")
backend = provider.backend(name="ibm_brisbane")

# create sampler from the actual backend
sampler = SamplerV2.from_backend(backend)

# run a sampler job on the parameterized circuits with noise model of the actual hardware
bell_t = transpile(bell, AerSimulator(basis_gates=["ecr", "id", "rz", "sx"]), optimization_level=0)
job3 = sampler.run([bell_t], shots=128)
job_result = job3.result()
print(f"counts for Bell circuit w/noise: {job_result[0].data.meas.get_counts()}")

qiskit_runtime_service._discover_account:WARNING:2025-11-04 09:30:34,883: Loading account with name main_school. Any input 'channel', 'token' or 'url' are ignored.


counts for Bell circuit w/noise: {'00': 65, '11': 57, '01': 4, '10': 2}
